## Import

In [1]:
import pandas as pd
import os

# Isso aqui limpa a "confusão" que o Python 3.13 está fazendo
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']
import matplotlib
# Isso avisa ao programa: "Ei, desenhe o gráfico aqui dentro do Jupyter mesmo!"
matplotlib.use('Agg')
from ImpararePackage import dataRequest

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, classification_report

from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
from sklearn import tree

import joblib

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
query = """
select 
    pa."caso infeccao",
    pa."caso infeccao hospitalar",
    pa."Conjuntivite",
    pa."Infecção da cavidade oral",
    pa."Infecção de pele",
    pa."Meningite",
    pa.comunitaria,
    pa.enterocolite,
    pa.ipcs,
    pa.isc,
    pa.pav,
    dtp.*
from imparare2_train_wellhead_set_prepared dtp
inner join new_avaliacao_padrao_ouro pa
    on dtp."REGISTRO" = pa.registro 
        and dtp."DIA_parsed" = pa.dt_infeccao
"""

df = dataRequest.get_data(queryText= query, chunck= None)

In [8]:
df.head()

,caso infeccao,caso infeccao hospitalar,Conjuntivite,Infecção da cavidade oral,Infecção de pele,Meningite,comunitaria,enterocolite,ipcs,isc,pav,REGISTRO,DIA_parsed,SEXO,IDADE,UNIDADE_INTERNACAO,ESPECIALIDADE_INTERNACAO,TIPO_INTERNACAO,CLINICA_INTERNACAO,dthr_internacao,dthr_alta,los_dias,los_horas,CIRURGIA_tempo_total,CIRURGIA_procedimentos_total,TECNICO(A) EM ENFERMAGEM_count,TECNICO(A) EM ENFERMAGEM_avg,ENFERMEIRO(A)_count,ENFERMEIRO(A)_avg,AUXILIAR DE ENFERMAGEM_count,AUXILIAR DE ENFERMAGEM_avg,ACADEMICODE ENFERMAGEM_count,ACADEMICO DE ENFERMAGEM_avg,MEDICO_count,MEDICO_avg,FISIOTERAPEUTA_count,FISIOTERAPEUTA_avg,OUTROS_count,OUTROS_avg,DIUR_min,DIUR_max,DIUR_count,DIUR_avg,DIUR_stddev,DIUR_delta,FC_min,FC_max,FC_ALTERADA,FC_ALTA,FC_BAIXA,FC_count,FC_avg,FC_stddev,FC_delta,FR_min,FR_max,FR_ALTERADA,FR_count,FR_avg,FR_stddev,FR_delta,HGT_min,HGT_max,HGT_ALTERADO,HGT_count,HGT_avg,HGT_stddev,HGT_delta,OXIMETRIA_min,OXIMETRIA_max,OXIMETRIA_count,OXIMETRIA_avg,OXIMETRIA_stddev,OXIMETRIA_delta,PAD_min,PAD_max,PAD_count,PAD_avg,PAD_stddev,PAD_delta,PAS_min,PAS_max,PAS_count,PAS_avg,PAS_stddev,PAS_delta,TEMP_min,TEMP_max,TEMP_ALTERADA,TEMP_ALTA,TEMP_BAIXA,TEMP_count,TEMP_avg,TEMP_stddev,TEMP_delta,PA_min,PA_max,PA_count,PA_avg,PA_stddev,PA_delta,PAM_min,PAM_max,PAM_count,PAM_avg,PAM_stddev,PAM_delta,O2_min,O2_max,O2_count,O2_avg,O2_stddev,O2_delta,FIO2_min,FIO2_max,FIO2_count,FIO2_avg,FIO2_stddev,FIO2_delta,PEEP_min,PEEP_max,PEEP_count,PEEP_avg,PEEP_stddev,PEEP_delta,SINAIS_VITAIS_registros_total,UNIDADE_fim_janela,UNIDADE_inicio_janela,UNIDADE_movimento_total,EVOLUCAO_notas_total,EVOLUCAO_notas_total_dia,aciclovir_count,amoxicilina_sulbactam_count,cefadroxila_count,cefalotina_count,cefotaxima_count,ceftazidima_count,ceftazidima_avibactam_count,ceftalozana_tazobactam_count,cetoconazol_count,claritromicina_count,cloranfenicol_count,doxiciclina_count,ertapenem_count,eritromicina_count,ganciclovir_count,ivermectina_count,micafungina_count,miconazol_count,moxifloxacino_count,mupirocina_count,neomicina_count,rifampicina_count,sulfadiazina_count,teicoplanina_count,voriconazol_count,ciprofloxacino_count,outro_count,amicacina_count,amoxicilina_clavulanato_count,amoxicilina_count,ampicilina_count,azitromicina_count,penicilinagbenzatina_count,penicilinagpotassica_count,cefalexina_count,cefazolina_count,cefepima_count,ceftriaxona_count,cefuroxima_count,clindamicina_count,fluconazol_count,gentamicina_count,levofloxacino_count,linezolida_count,meropenem_count,metronidazol_count,nistatina_count,oxacilina_count,piperacilina_tazobactam_count,ampicilina_sulbactam_count,sulfametoxazol_trimetoprim_count,polimixinab_count,tobramicina_count,vancomicina_count,via_topico_count,via_sng_count,via_outro_count,via_ev_count,via_im_count,via_vo_count,agg_hemocultura,qtd_hemocultura_positiva,qtd_hemocultura_sem_crescimento,qtd_cultura_positiva,qtd_cultura_sem_crescimento,qtd_bacteroscopico_positiva,qtd_bacteroscopico_negativa,agg_urocultura,qtd_urocultura_positiva,qtd_urocultura_sem_crescimento,qtd_bacteriologico_positiva,qtd_bacteriologico_sem_crescimento,leuco_min,leuco_max,leuco_avg,rdw_min,rdw_max,rdw_avg,neuto_min,neuto_max,neuto_avg,LEUCO_ALTERADO,LEUCO_ALTO,LEUCO_BAIXO,RDW_ALTO,NEUTROFILO_ALTO,RAGIOLOGIA_laudos_total,tosse,ronco,cavita,opac,infil,fibrose_cistica,sec_pur_pulmonar,secr,puru,secr_traq,o2,dpoc,hemoptise,leucemia,esplenectomia,febre,consol,linfoma,acinetobacter,amarelada,aspiracao,broncograma_aereo,crepitante,enterobacter,enterococcus,hemophylus,hipertermia,imunossuprimido,infeccao,klebsiella,legionella,leucocitose,leucopenia,moraxella,pneumococcus,pseudomonas,sibilos,intubacao,staphylococcus_aureus,sepse_pulmonar,sepse_respiratoria,sepse_urinaria,padrao_ventilatorio,insuficiencia_ventilatoria,esforco_ventilatorio,lavado_broncoalveolar,derrame_pleural,escherichia_coli,choque_septico,dor_pleuritica,dor_ventilatorio_dependente,ventilacao_mecanica,iot,pav,endoftalmite,eritema,hiperemia,mediastinite,staphylococcus_coagulase_negativo,supuracao,ru

In [4]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 29292 entries, 0 to 29291
Columns: 712 entries, caso infeccao to UNIDADE_fim_janela_Gineco-obstetrícia
dtypes: bool(2), datetime64[us](3), float64(183), int64(52), object(9), str(463)
memory usage: 501.8 MB


In [5]:
df.memory_usage(deep=True)

Index                                                   132
caso infeccao                                         29292
caso infeccao hospitalar                              29292
Conjuntivite                                         770760
Infecção da cavidade oral                            770760
Infecção de pele                                     770760
Meningite                                            770760
comunitaria                                          770760
enterocolite                                         770760
ipcs                                                 770760
isc                                                  770760
pav                                                  770760
REGISTRO                                             234336
DIA_parsed                                           234336
SEXO                                                1464600
IDADE                                                234336
UNIDADE_INTERNACAO                      

### Export de dataset

In [ ]:
# df.to_feather("dataset_novos_criterios_neo_completo")

## Organização de variáveis

In [11]:
# Dado controle - quem é e de quando é
dados_controle = ["REGISTRO", "DIA_parsed", "dthr_internacao", "dthr_alta"]

# Dado texto: 
dado_categoricos = ["UNIDADE_INTERNACAO", "ESPECIALIDADE_INTERNACAO", "TIPO_INTERNACAO", "CLINICA_INTERNACAO", "SEXO", "TEMP_INCUBADORA_min", "TEMP_INCUBADORA_max", "TEMP_INCUBADORA_count", "TEMP_INCUBADORA_avg", "TEMP_INCUBADORA_stddev", "TEMP_INCUBADORA_delta", "UNIDADE_fim_janela", "UNIDADE_inicio_janela", "agg_hemocultura", "agg_urocultura"]

# Gabaritos
gabaritos = ["caso_infeccao", "pnm", "isc", "itu", "pav", "ipcs", "traqueobronquite"]
gabaritos = ['caso infeccao', 'caso infeccao hospitalar', 'Conjuntivite', 'Infecção da cavidade oral', 'Infecção de pele', 'Meningite', 'comunitaria', 'enterocolite', 'ipcs', 'isc', 'pav']

# Features
features_atendimento = ["IDADE", "los_dias", "los_horas"]

features_cirurgias = ["CIRURGIA_tempo_total", "CIRURGIA_procedimentos_total"]

# Sinais vitais
features_sinal_vital = ["TECNICO(A) EM ENFERMAGEM_count", "TECNICO(A) EM ENFERMAGEM_avg", "ENFERMEIRO(A)_count", "ENFERMEIRO(A)_avg", "AUXILIAR DE ENFERMAGEM_count", "AUXILIAR DE ENFERMAGEM_avg", "ACADEMICODE ENFERMAGEM_count", "ACADEMICO DE ENFERMAGEM_avg", "MEDICO_count", "MEDICO_avg", "FISIOTERAPEUTA_count", "FISIOTERAPEUTA_avg", "OUTROS_count", "OUTROS_avg", "DIUR_min", "DIUR_max", "DIUR_count", "DIUR_avg", "DIUR_stddev", "DIUR_delta", "FC_min", "FC_max", "FC_ALTERADA", "FC_ALTA", "FC_BAIXA", "FC_count", "FC_avg", "FC_stddev", "FC_delta", "FR_min", "FR_max", "FR_ALTERADO", "FR_count", "FR_avg", "FR_stddev", "FR_delta", "HGT_min", "HGT_max", "HGT_ALTERADO", "HGT_count", "HGT_avg", "HGT_stddev", "HGT_delta", "OXIMETRIA_min", "OXIMETRIA_max", "OXIMETRIA_count", "OXIMETRIA_avg", "OXIMETRIA_stddev", "OXIMETRIA_delta", "PAD_min", "PAD_max", "PAD_count", "PAD_avg", "PAD_stddev", "PAD_delta", "PAS_min", "PAS_max", "PAS_count", "PAS_avg", "PAS_stddev", "PAS_delta", "TEMP_min", "TEMP_max", "TEMP_ALTERADA", "TEMP_ALTA", "TEMP_BAIXA", "TEMP_count", "TEMP_avg", "TEMP_stddev", "TEMP_delta", "PA_min", "PA_max", "PA_count", "PA_avg", "PA_stddev", "PA_delta", "PAM_min", "PAM_max", "PAM_count", "PAM_avg", "PAM_stddev", "PAM_delta", "O2_min", "O2_max", "O2_count", "O2_avg", "O2_stddev", "O2_delta", "FIO2_min", "FIO2_max", "FIO2_count", "FIO2_avg", "FIO2_stddev", "FIO2_delta", "PEEP_min", "PEEP_max", "PEEP_count", "PEEP_avg", "PEEP_stddev", "PEEP_delta", "SINAIS_VITAIS_registros_total"]

# Volumetrico evos e movimentacao
features_evol = ["UNIDADE_movimento_total", "EVOLUCAO_notas_total", "EVOLUCAO_notas_total_dia"]

# Antibioticos utilizados
features_atb = ["aciclovir_count", "amoxicilina_sulbactam_count", "cefadroxila_count", "cefalotina_count", "cefotaxima_count", "ceftazidima_count", "ceftazidima_avibactam_count", "ceftalozana_tazobactam_count", "cetoconazol_count", "claritromicina_count", "cloranfenicol_count", "doxiciclina_count", "ertapenem_count", "eritromicina_count", "ganciclovir_count", "ivermectina_count", "micafungina_count", "miconazol_count", "moxifloxacino_count", "mupirocina_count", "neomicina_count", "rifampicina_count", "sulfadiazina_count", "teicoplanina_count", "voriconazol_count", "ciprofloxacino_count", "outro_count", "amicacina_count", "amoxicilina_clavulanato_count", "amoxicilina_count", "ampicilina_count", "azitromicina_count", "penicilinagbenzatina_count", "penicilinagpotassica_count", "cefalexina_count", "cefazolina_count", "cefepima_count", "ceftriaxona_count", "cefuroxima_count", "clindamicina_count", "fluconazol_count", "gentamicina_count", "levofloxacino_count", "linezolida_count", "meropenem_count", "metronidazol_count", "nistatina_count", "oxacilina_count", "piperacilina_tazobactam_count", "ampicilina_sulbactam_count", "sulfametoxazol_trimetoprim_count", "polimixinab_count", "tobramicina_count", "vancomicina_count", "via_topico_count", "via_sng_count", "via_outro_count", "via_ev_count", "via_im_count", "via_vo_count"]

# Dados de exame
# Culturais
features_culturais = ["qtd_hemocultura_positiva", "qtd_hemocultura_sem_crescimento", "qtd_cultura_positiva", "qtd_cultura_sem_crescimento", "qtd_bacteroscopico_positiva", "qtd_bacteroscopico_negativa", "qtd_urocultura_positiva", "qtd_urocultura_sem_crescimento", "qtd_bacteriologico_positiva", "qtd_bacteriologico_sem_crescimento"]

# Sangue
features_sangue = ["leuco_min", "leuco_max", "leuco_avg", "rdw_min", "rdw_max", "rdw_avg", "neuto_min", "neuto_max", "neuto_avg", "LEUCO_ALTERADO", "LEUCO_ALTO", "LEUCO_BAIXO", "RDW_ALTO", "NEUTROFILO_ALTO"]

# Laudos Imagem
features_laudos_imagem = ["RAGIOLOGIA_laudos_total"]

# Proporção de termos no texto 
features_proporcao_texto = ["tosse", "ronco", "cavita", "opac", "infil", "fibrose_cistica", "sec_pur_pulmonar", "secr", "puru", "secr_traq", "o2", "hemoptise", "leucemia", "esplenectomia", "febre", "consol", "linfoma", "acinetobacter", "amarelada", "aspiracao", "crepitante", "enterobacter", "enterococcus", "hemophylus", "hipertermia", "imunossuprimido", "infeccao", "klebsiella", "leucocitose", "leucopenia", "moraxella", "pneumococcus", "pseudomonas", "sibilos", "intubacao", "staphylococcus_aureus", "sepse_pulmonar", "sepse_respiratoria", "sepse_urinaria", "padrao_ventilatorio", "insuficiencia_ventilatoria", "esforco_ventilatorio", "lavado_broncoalveolar", "derrame_pleural", "escherichia_coli", "choque_septico", "dor_pleuritica", "dor_ventilatorio_dependente", "ventilacao_mecanica", "iot", 
                            # "pav_1",
                            "eritema", "hiperemia", "mediastinite", "staphylococcus_coagulase_negativo", "supuracao", "rubor", "mal_estar", "abscesso", "calor", "dreno", "edema", "endocardite", "antibiotico", "bacteriologico", "deiscencia", "resistente", "taquicardia", "cateter", "cateter_arterial", "cateter_urinario", "cvc", "cateter_venoso_periferico", "instabilidade", "oliguria", "hipotensao", "anuria", "bacteremia", "bacteriuria", "disuria", "leucocituria", "nitrito", "svd", "tsa", "choque", "staphylococcus", "e_coli", "colecao", "tremor", "calafrio", "pos_operatorio", "dor_suprapubica", "foco_urinario", "colite_pseudomembranosa", "vomito", "shigella", "salmonela", "nausea", "dor_cabeca", "dor_abdominal", "diarreia", "clostridium", "urgencia_urinaria", "urgencia_miccional", "sondagem_alivio", "sondagem_vesical", "urocultura", "piuria", "albumina", "press_parc_co2", "creatinina", "glicose", "plaquetas", "bilirrubina", "cardiomegalia", "lesao_pulmonar", "pneumonia", "atelectasia", "pneumotorax", "fratura", "freq_respiratoria", "freq_cardiaca", "pressao_sanguinea", "saturacao_sangue", "hemocultura", "temperatura", "sibilancia", "hipotermia", "apneia", "bradicardia", "streptococcus_viridans", "consciencia", "hcm", "hgm", "covid", "peep", "fio2", "spo2", "sne", "desnutricao", "sedacao", "vsg", "fosfatase_alcalina", "ldh", "cetamina", "propofol", "clorpromazina", "fentanil", "pancuronio", "noradrenalina", "diazepan", "lorazepan", "flumazenil", "clonidina", "npt", "cateter_monolumen", "cateter_duplolumen", "cateter_triplolumen", "portocath", "cateter_hickmann", "shilley", "obesidade", "diabetes", "neutropenia"]

# contagem de termos em texto
features_contagem_texto = ["tosse_pos_count", "ronco_pos_count", "cavita_pos_count", "opac_pos_count", "infil_pos_count", "fibrose_cistica_pos_count", "sec_pur_pulmonar_pos_count", "secr_pos_count", "puru_pos_count", "secr_traq_pos_count", "o2_pos_count", "hemoptise_pos_count", "leucemia_pos_count", "esplenectomia_pos_count", "febre_pos_count", "consol_pos_count", "linfoma_pos_count", "acinetobacter_pos_count", "amarelada_pos_count", "aspiracao_pos_count", "crepitante_pos_count", "enterobacter_pos_count", "enterococcus_pos_count", "hemophylus_pos_count", "hipertermia_pos_count", "imunossuprimido_pos_count", "infeccao_pos_count", "klebsiella_pos_count", "leucocitose_pos_count", "leucopenia_pos_count", "moraxella_pos_count", "pneumococcus_pos_count", "pseudomonas_pos_count", "sibilos_pos_count", "intubacao_pos_count", "staphylococcus_aureus_pos_count", "sepse_pulmonar_pos_count", "sepse_respiratoria_pos_count", "sepse_urinaria_pos_count", "padrao_ventilatorio_pos_count", "insuficiencia_ventilatoria_pos_count", "esforco_ventilatorio_pos_count", "lavado_broncoalveolar_pos_count", "derrame_pleural_pos_count", "escherichia_coli_pos_count", "choque_septico_pos_count", "dor_pleuritica_pos_count", "dor_ventilatorio_dependente_pos_count", "ventilacao_mecanica_pos_count", "iot_pos_count", "pav_pos_count", "eritema_pos_count", "hiperemia_pos_count", "mediastinite_pos_count", "staphylococcus_coagulase_negativo_pos_count", "supuracao_pos_count", "rubor_pos_count", "mal_estar_pos_count", "abscesso_pos_count", "calor_pos_count", "dreno_pos_count", "edema_pos_count", "endocardite_pos_count", "antibiotico_pos_count", "bacteriologico_pos_count", "deiscencia_pos_count", "resistente_pos_count", "taquicardia_pos_count", "cateter_pos_count", "cateter_arterial_pos_count", "cateter_urinario_pos_count", "cvc_pos_count", "cateter_venoso_periferico_pos_count", "instabilidade_pos_count", "oliguria_pos_count", "hipotensao_pos_count", "anuria_pos_count", "bacteremia_pos_count", "bacteriuria_pos_count", "disuria_pos_count", "leucocituria_pos_count", "nitrito_pos_count", "svd_pos_count", "tsa_pos_count", "choque_pos_count", "staphylococcus_pos_count", "e_coli_pos_count", "colecao_pos_count", "tremor_pos_count", "calafrio_pos_count", "pos_operatorio_pos_count", "dor_suprapubica_pos_count", "foco_urinario_pos_count", "colite_pseudomembranosa_pos_count", "vomito_pos_count", "shigella_pos_count", "salmonela_pos_count", "nausea_pos_count", "dor_cabeca_pos_count", "dor_abdominal_pos_count", "diarreia_pos_count", "clostridium_pos_count", "urgencia_urinaria_pos_count", "urgencia_miccional_pos_count", "sondagem_alivio_pos_count", "sondagem_vesical_pos_count", "urocultura_pos_count", "piuria_pos_count", "albumina_pos_count", "press_parc_co2_pos_count", "creatinina_pos_count", "glicose_pos_count", "plaquetas_pos_count", "bilirrubina_pos_count", "cardiomegalia_pos_count", "lesao_pulmonar_pos_count", "pneumonia_pos_count", "atelectasia_pos_count", "pneumotorax_pos_count", "fratura_pos_count", "freq_respiratoria_pos_count", "freq_cardiaca_pos_count", "pressao_sanguinea_pos_count", "saturacao_sangue_pos_count", "hemocultura_pos_count", "temperatura_pos_count", "sibilancia_pos_count", "hipotermia_pos_count", "apneia_pos_count", "bradicardia_pos_count", "streptococcus_viridans_pos_count", "consciencia_pos_count", "hcm_pos_count", "hgm_pos_count", "covid_pos_count", "peep_pos_count", "fio2_pos_count", "spo2_pos_count", "sne_pos_count", "desnutricao_pos_count", "sedacao_pos_count", "vsg_pos_count", "fosfatase_alcalina_pos_count", "ldh_pos_count", "cetamina_pos_count", "propofol_pos_count", "clorpromazina_pos_count", "fentanil_pos_count", "pancuronio_pos_count", "noradrenalina_pos_count", "diazepan_pos_count", "lorazepan_pos_count", "flumazenil_pos_count", "clonidina_pos_count", "npt_pos_count", "cateter_monolumen_pos_count", "cateter_duplolumen_pos_count", "cateter_triplolumen_pos_count", "portocath_pos_count", "cateter_hickmann_pos_count", "shilley_pos_count", "obesidade_pos_count", "diabetes_pos_count", "neutropenia_pos_count"]

#Critérios adultos que a Stephani tirou
features_adulto_antigo = ["endoftalmite", "propionibacterium", "avc", "cryptococcus", "bicarbonato", "pneumocystis", "corynecterium", "broncograma_aereo", "calcio", "histoplasma", "bacillus", "yersinia", "paracoccidioides", "aerococcus", "bacteroides", "clorose", "coccidioides", "hemoglobina", "candida", "veilonella", "potassio", "fusobacterium", "giardia", "sodio", "peptostreptococcus", "dpoc", "legionella", "tabagismo", "campylobacter", "ph_sangue"]

# Proporção de termos no texto - novos critérios
features_proporcao_texto_novos = ["pneumatose", "piora_troca_gasosa", "fontanela_abaulada", "glicose_liquor", "prote_liquor", "leuco_liquor", "bacteria_liquor", "irritabilidade", "cultura_liquor", "hipoativ", "dren_serosa", "necrose_intestinal", "intestino_fixo", "sangue_fezes", "convulsao", "pneumoperitonio", "dist_abdominal", "asp_bilioso", "pcr","letargia", "int_glicose", "int_alimentar"]
        
# contagem de termos em texto - novos critérios
features_contagem_texto_novos = ["pneumatose_pos_count", "piora_troca_gasosa_pos_count", "fontanela_abaulada_pos_count", "glicose_liquor_pos_count", "prote_liquor_pos_count", "leuco_liquor_pos_count", "bacteria_liquor_pos_count", "irritabilidade_pos_count", "cultura_liquor_pos_count", "hipoativ_pos_count", "dren_serosa_pos_count", "necrose_intestinal_pos_count", "intestino_fixo_pos_count", "sangue_fezes_pos_count", "convulsao_pos_count", "pneumoperitonio_pos_count", "dist_abdominal_pos_count", "asp_bilioso_pos_count", "pcr_pos_count","letargia_pos_count", "int_glicose_pos_count", "int_alimentar_pos_count"]


### Gabarito

In [12]:
df_texto_gabarito = df[gabaritos+features_culturais+features_sangue+features_proporcao_texto+features_proporcao_texto_novos]
df_texto_gabarito.head()

,caso infeccao,caso infeccao hospitalar,Conjuntivite,Infecção da cavidade oral,Infecção de pele,Meningite,comunitaria,enterocolite,ipcs,isc,pav,pav,qtd_hemocultura_positiva,qtd_hemocultura_sem_crescimento,qtd_cultura_positiva,qtd_cultura_sem_crescimento,qtd_bacteroscopico_positiva,qtd_bacteroscopico_negativa,qtd_urocultura_positiva,qtd_urocultura_sem_crescimento,qtd_bacteriologico_positiva,qtd_bacteriologico_sem_crescimento,leuco_min,leuco_max,leuco_avg,rdw_min,rdw_max,rdw_avg,neuto_min,neuto_max,neuto_avg,LEUCO_ALTERADO,LEUCO_ALTO,LEUCO_BAIXO,RDW_ALTO,NEUTROFILO_ALTO,tosse,ronco,cavita,opac,infil,fibrose_cistica,sec_pur_pulmonar,secr,puru,secr_traq,o2,hemoptise,leucemia,esplenectomia,febre,consol,linfoma,acinetobacter,amarelada,aspiracao,crepitante,enterobacter,enterococcus,hemophylus,hipertermia,imunossuprimido,infeccao,klebsiella,leucocitose,leucopenia,moraxella,pneumococcus,pseudomonas,sibilos,intubacao,staphylococcus_aureus,sepse_pulmonar,sepse_respiratoria,sepse_urinaria,padrao_ventilatorio,insuficiencia_ventilatoria,esforco_ventilatorio,lavado_broncoalveolar,derrame_pleural,escherichia_coli,choque_septico,dor_pleuritica,dor_ventilatorio_dependente,ventilacao_mecanica,iot,eritema,hiperemia,mediastinite,staphylococcus_coagulase_negativo,supuracao,rubor,mal_estar,abscesso,calor,dreno,edema,endocardite,antibiotico,bacteriologico,deiscencia,resistente,taquicardia,cateter,cateter_arterial,cateter_urinario,cvc,cateter_venoso_periferico,instabilidade,oliguria,hipotensao,anuria,bacteremia,bacteriuria,disuria,leucocituria,nitrito,svd,tsa,choque,staphylococcus,e_coli,colecao,tremor,calafrio,pos_operatorio,dor_suprapubica,foco_urinario,colite_pseudomembranosa,vomito,shigella,salmonela,nausea,dor_cabeca,dor_abdominal,diarreia,clostridium,urgencia_urinaria,urgencia_miccional,sondagem_alivio,sondagem_vesical,urocultura,piuria,albumina,press_parc_co2,creatinina,glicose,plaquetas,bilirrubina,cardiomegalia,lesao_pulmonar,pneumonia,atelectasia,pneumotorax,fratura,freq_respiratoria,freq_cardiaca,pressao_sanguinea,saturacao_sangue,hemocultura,temperatura,sibilancia,hipotermia,apneia,bradicardia,streptococcus_viridans,consciencia,hcm,hgm,covid,peep,fio2,spo2,sne,desnutricao,sedacao,vsg,fosfatase_alcalina,ldh,cetamina,propofol,clorpromazina,fentanil,pancuronio,noradrenalina,diazepan,lorazepan,flumazenil,clonidina,npt,cateter_monolumen,cateter_duplolumen,cateter_triplolumen,portocath,cateter_hickmann,shilley,obesidade,diabetes,neutropenia,pneumatose,piora_troca_gasosa,fontanela_abaulada,glicose_liquor,prote_liquor,leuco_liquor,bacteria_liquor,irritabilidade,cultura_liquor,hipoativ,dren_serosa,necrose_intestinal,intestino_fixo,sangue_fezes,convulsao,pneumoperitonio,dist_abdominal,asp_bilioso,pcr,letargia,int_glicose,int_alimentar
0,False,False,False,False,False,False,False,False,False,False,False,NaN,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,False,False,False,False,False,False,False,False,False,False,False,NaN,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

### Variáveis usadas - texto + exames + laudos

In [25]:
X_texto = df_texto_gabarito.drop(columns= gabaritos)
y_texto = df_texto_gabarito['caso infeccao']
X_texto_train, X_texto_test, y_texto_train, y_texto_test = train_test_split(X_texto, y_texto, test_size=0.2, random_state=42, stratify= y_texto)
len(X_texto_train), len(X_texto_test), len(y_texto_train), len(y_texto_test)

(23433, 5859, 23433, 5859)

# Data Exploring

## Split dataset

In [26]:
# X = df.drop(columns= ['REGISTRO', 'los_horas', 'caso_infeccao', 'pnm', 'isc', 'itu', 'pav', 'ipcs', 'traqueobronquite', 'DIA_parsed', 'SEXO', 'UNIDADE_INTERNACAO', 'UNIDADE_INTERNACAO', 'ESPECIALIDADE_INTERNACAO', 'TIPO_INTERNACAO',	'CLINICA_INTERNACAO', 'dthr_internacao', 'dthr_alta', 'UNIDADE_fim_janela', 'UNIDADE_inicio_janela', 'agg_hemocultura', 'agg_urocultura'])
X = df.drop(columns= gabaritos)
y = df['caso infeccao']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify= y)

## Decision Tree

### Best Estimator trial

In [20]:
#Modelo base
dt = DecisionTreeClassifier(random_state= 28)

# Grid de parâmetros para testar
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],            # medida de impureza
    'max_depth': list(range(10, 11)) + [None],        # profundidade máxima
    'min_samples_split': list(range(3, 20)),         # mín. amostras para dividir um nó
    'min_samples_leaf': list(range(2, 20)),            # mín. amostras em uma folha
    'max_features': [None],      # nº de features consideradas por vez
    'splitter': ['best', 'random'],
    'ccp_alpha': [0, 0.001, 0.002, 0.005, 0.01, 0.02],
    'class_weight': [None, "balanced"]
}

scorer = 'recall'

scoring_metrics = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# GridSearch
grid = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=5,
    scoring=scoring_metrics,
    refit=scorer,
    n_jobs=-1,
    verbose=3,
    return_train_score=False
)

# Treinar o grid
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
joblib.dump(best_model, 'melhor_modelo_recall_focado.pkl')

print(f"\nMelhor modelo salvo (Decidido pelo Recall).")
print(f"Melhores parâmetros: {grid.best_params_}")
# Note que agora o best_score_ se refere à métrica do refit
print(f"Melhor Recall: {grid.best_score_:.4f}")

# 5. Salvando o DataFrame Completo (Agora muito mais rico)
results_df = pd.DataFrame(grid.cv_results_)

# As colunas agora terão prefixos com os nomes das métricas
# Ex: mean_test_recall, mean_test_precision, mean_test_f1
cols_to_keep = [
    'params', 
    'mean_test_recall', 
    'mean_test_precision', 
    'mean_test_f1',
    'mean_test_roc_auc'
]

# Ordenando pelo Recall (ou a métrica que você preferir visualizar)
final_df = results_df[cols_to_keep].sort_values(by='mean_test_recall', ascending=False)

final_df.to_csv('grid_multimetric_results.csv', index=False)
print("CSV salvo com todas as métricas comparadas!")

final_df = results_df.sort_values(by='mean_test_recall', ascending=False)

final_df.to_csv('grid_multimetric_results_completo.csv', index=False)

Fitting 5 folds for each of 44064 candidates, totalling 220320 fits
[CV 1/5] END ccp_alpha=0, class_weight=None, criterion=gini, max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=3, splitter=best; f1: (test=nan) precision: (test=nan) recall: (test=nan) roc_auc: (test=nan) total time=   0.3s
[CV 2/5] END ccp_alpha=0, class_weight=None, criterion=gini, max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=3, splitter=best; f1: (test=nan) precision: (test=nan) recall: (test=nan) roc_auc: (test=nan) total time=   0.3s
[CV 3/5] END ccp_alpha=0, class_weight=None, criterion=gini, max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=3, splitter=best; f1: (test=nan) precision: (test=nan) recall: (test=nan) roc_auc: (test=nan) total time=   0.3s
[CV 4/5] END ccp_alpha=0, class_weight=None, criterion=gini, max_depth=10, max_features=None, min_samples_leaf=2, min_samples_split=3, splitter=best; f1: (test=nan) precision: (test=nan) recall: (t

KeyboardInterrupt: 

In [44]:
model = DecisionTreeClassifier(
    criterion='gini', 
    splitter='random', 
    random_state= 113, 
    min_samples_leaf= 4,
    min_samples_split= 4,
    max_features= None,
    max_depth= None,
    class_weight= 'balanced'# {0: 1, 1: 20}
)

model.fit(X_texto_train, y_texto_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'random'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",4
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",113
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curr

In [45]:
y_train_pred = model.predict(X_texto_train)
acc = accuracy_score(y_train, y_train_pred)
print(f"Accurácia: {acc}")
cm = confusion_matrix(y_train, y_train_pred)
print(f"Confusion Matrix treino: \n{cm}\n")
report = classification_report(y_train, y_train_pred)
print(report)

print("\n=========================================================\n")
# Validando test
y_pred = model.predict(X_texto_test)
    
acc = accuracy_score(y_test, y_pred)
print(f"Accurácia: {acc}")
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix teste: \n{cm}")
report = classification_report(y_test, y_pred)
print(report)

Accurácia: 0.9318482481969872
Confusion Matrix treino: 
[[21522  1527]
 [   70   314]]

              precision    recall  f1-score   support

       False       1.00      0.93      0.96     23049
        True       0.17      0.82      0.28       384

    accuracy                           0.93     23433
   macro avg       0.58      0.88      0.62     23433
weighted avg       0.98      0.93      0.95     23433



Accurácia: 0.9300221880867042
Confusion Matrix teste: 
[[5382  381]
 [  29   67]]
              precision    recall  f1-score   support

       False       0.99      0.93      0.96      5763
        True       0.15      0.70      0.25        96

    accuracy                           0.93      5859
   macro avg       0.57      0.82      0.60      5859
weighted avg       0.98      0.93      0.95      5859



## Analisando importância dos parâmetros

In [46]:
pd.options.display.float_format = '{:.4f}'.format
importances = pd.Series(model.feature_importances_, index=X_texto_test.columns)

# Ordena em ordem decrescente
importances = importances.sort_values(ascending=False)
print(importances)

importances = importances.to_dict()
importances['model'] = 'random_forest'

qtd_cultura_positiva                 0.3676
qtd_hemocultura_sem_crescimento      0.3480
qtd_cultura_sem_crescimento          0.1539
LEUCO_ALTERADO                       0.0358
neuto_min                            0.0154
leuco_avg                            0.0099
dreno                                0.0094
int_glicose                          0.0082
leuco_max                            0.0067
rdw_min                              0.0058
neuto_avg                            0.0052
leuco_min                            0.0051
sedacao                              0.0049
rdw_max                              0.0046
neuto_max                            0.0038
LEUCO_BAIXO                          0.0030
rdw_avg                              0.0029
qtd_urocultura_sem_crescimento       0.0025
RDW_ALTO                             0.0019
sangue_fezes                         0.0018
NEUTROFILO_ALTO                      0.0011
spo2                                 0.0010
qtd_urocultura_positiva         

In [47]:
text_tree = tree.export_text(model, feature_names=X_texto.columns)
print(text_tree)

|--- qtd_hemocultura_sem_crescimento <= 0.52
|   |--- qtd_cultura_positiva <= 1.09
|   |   |--- LEUCO_ALTERADO <= 0.06
|   |   |   |--- RDW_ALTO <= 0.41
|   |   |   |   |--- qtd_cultura_sem_crescimento <= 0.84
|   |   |   |   |   |--- leuco_min <= 19689.68
|   |   |   |   |   |   |--- neuto_avg <= 2073.63
|   |   |   |   |   |   |   |--- class: False
|   |   |   |   |   |   |--- neuto_avg >  2073.63
|   |   |   |   |   |   |   |--- leuco_avg <= 15573.79
|   |   |   |   |   |   |   |   |--- neuto_min <= 8714.82
|   |   |   |   |   |   |   |   |   |--- rdw_min <= 14.97
|   |   |   |   |   |   |   |   |   |   |--- neuto_min <= 6988.90
|   |   |   |   |   |   |   |   |   |   |   |--- truncated branch of depth 4
|   |   |   |   |   |   |   |   |   |   |--- neuto_min >  6988.90
|   |   |   |   |   |   |   |   |   |   |   |--- class: False
|   |   |   |   |   |   |   |   |   |--- rdw_min >  14.97
|   |   |   |   |   |   |   |   |   |   |--- leuco_avg <= 11816.93
|   |   |   |   |   |   |   | 